# P1M-A1 part B: Taiwan's tagged rules with feedback decoupled from flow persistence (2013–2016)

Analysis fixed before the run in `P1M_A1_protocol.md`. It reuses the TWSE files already on your Drive in `P1T_raw` (development period only, 2013-01-02 to 2016-12-30; nothing later is loaded) with P1M's data code, unchanged. It estimates each class's response to the price deviation as a free distributed lag, with the class's other flow following its own autoregression, and computes each class's contribution to the feedback budget with a date-block bootstrap.

**Run the single code cell once** (Runtime → Run all). Allow Drive access when asked. Loading the files takes most of the time (about 15–30 minutes). It saves `p1m_a1_partB_results.json` and `p1m_a1_partB_results.zip` to Drive (`P1T_raw`) and downloads the zip. Send the zip, or paste the printed summary. A rerun loads the saved result and does not recompute.

In [ ]:

import os, io, json, gzip, time, zipfile, hashlib, datetime, re, sys, calendar
import numpy as np, requests
EXECUTOR_SHA256 = "8d8a70a36f412591c03dfeaec463a0cb949aa4df565fdad829b984462615b075"
CORE_SHA256 = "50df914b54482eb3ffef2c26a4da1066fb08778a03f4caef052f116d8f0310aa"
EXECUTOR_SRC = "\"\"\"P1T executor: Taiwan tagged-flow forecast-gain test (registered spec P1T_taiwan_spec.md).\n\nLayers\n------\n1. Core (pure numpy): per-stock forecast losses, the primary statistic D, and the stationary date bootstrap.\n2. Data: parse stored TWSE T86 and MI_INDEX JSON responses into a panel, apply the universe and exclusion rules,\n   and build u (class flows), x (market-reference basis) and the limit indicator.\n3. Run: one period at a time (\"validation\" or \"holdout\"), writing a results JSON.\n\nThe executor never downloads. Downloads are done by the Colab notebook, which stores raw responses with SHA-256.\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport math\nimport os\nimport re\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nD_STAR = 0.0035\nSEED = 20261107\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# 1. Core\n# ----------------------------------------------------------------------------------------------------------------\n\ndef _design(u, x, lim, rows, with_x):\n    \"\"\"Rows index the forecast origin t; target is u[t+1].\"\"\"\n    cols = [np.ones(len(rows)), u[rows], u[rows - 1], lim[rows]]\n    if with_x:\n        cols += [x[rows], x[rows - 1]]\n    return np.column_stack(cols)\n\n\ndef stock_losses(u, x, lim, usable, fit_mask, eval_mask):\n    \"\"\"Per-date loss differential for one stock.\n\n    u, x, lim, usable: arrays over the full date axis. usable[t] means day t may serve as a forecast origin\n    (t-1, t, t+1 valid and x_t, x_{t-1} defined). fit_mask / eval_mask select origins for fitting / evaluation.\n    Returns (ell, var_fit) where ell[t] = (e0^2 - e1^2) / var_fit on evaluation origins and NaN elsewhere.\n    \"\"\"\n    T = len(u)\n    t_all = np.arange(1, T - 1)\n    fin = np.isfinite(u)\n    # Amendment A1: an origin needs finite flows at t-1, t, t+1 (a no-op for Foreign; needed for classes that a\n    # T86 schema era does not report).\n    ok = usable[1:T - 1] & fin[0:T - 2] & fin[1:T - 1] & fin[2:T]\n    fit = t_all[ok & fit_mask[1:T - 1]]\n    ev = t_all[ok & eval_mask[1:T - 1]]\n    ell = np.full(T, np.nan)\n    if len(fit) < 50 or len(ev) < 50:\n        return ell, np.nan\n    y_fit = u[fit + 1]\n    var_fit = float(np.var(y_fit))\n    if not np.isfinite(var_fit) or var_fit <= 0:\n        return ell, np.nan\n    b0 = np.linalg.lstsq(_design(u, x, lim, fit, False), y_fit, rcond=None)[0]\n    b1 = np.linalg.lstsq(_design(u, x, lim, fit, True), y_fit, rcond=None)[0]\n    e0 = u[ev + 1] - _design(u, x, lim, ev, False) @ b0\n    e1 = u[ev + 1] - _design(u, x, lim, ev, True) @ b1\n    ell[ev] = (e0 ** 2 - e1 ** 2) / var_fit\n    return ell, var_fit\n\n\ndef panel_D(L):\n    \"\"\"L: stocks x dates, NaN where not an evaluation origin. D = mean over stocks of each stock's mean loss gain.\"\"\"\n    per = np.nanmean(L, axis=1)\n    per = per[np.isfinite(per)]\n    return float(per.mean()), per\n\n\ndef stationary_indices(n, mean_block, rng):\n    p = 1.0 / mean_block\n    idx = np.empty(n, dtype=np.int64)\n    idx[0] = rng.integers(n)\n    for i in range(1, n):\n        idx[i] = rng.integers(n) if rng.random() < p else (idx[i - 1] + 1) % n\n    return idx\n\n\ndef bootstrap_D(L, mean_block=20, B=9999, seed=SEED):\n    \"\"\"Stationary bootstrap over dates, the same resampled dates for every stock.\"\"\"\n    cols = np.where(np.isfinite(L).any(axis=0))[0]\n    Lc = L[:, cols]\n    rng = np.random.default_rng(seed)\n    n = Lc.shape[1]\n    S = np.nan_to_num(Lc)\n    C = np.isfinite(Lc).astype(float)\n    out = np.empty(B)\n    for b in range(B):\n        idx = stationary_indices(n, mean_block, rng)\n        num = S[:, idx].sum(1)\n        cnt = C[:, idx].sum(1)\n        per = num[cnt > 0] / cnt[cnt > 0]\n        out[b] = per.mean()\n    return out\n\n\ndef decide(D, boot):\n    \"\"\"Registered criteria. One-sided p for D > 0 from the bootstrap distribution centred at D.\"\"\"\n    lo, hi = np.percentile(boot, [2.5, 97.5])\n    p = float(np.mean(boot - D >= D)) if D > 0 else 1.0\n    if p < 0.05 and D > 0 and lo > 0:\n        verdict = \"supportive\"\n    elif hi < D_STAR:\n        verdict = \"inconsistent\"\n    else:\n        verdict = \"inconclusive\"\n    return {\"D\": D, \"p_one_sided\": p, \"ci95\": [float(lo), float(hi)], \"D_star\": D_STAR, \"verdict\": verdict}\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# 2. Data\n# ----------------------------------------------------------------------------------------------------------------\n\n# Amendment A1 (7 November 2026, before any statistic was computed). TWSE's T86 report has three schema eras in the\n# registered sample (census of every stored response, schema_census.json):\n#   E1 2013-01-02 .. 2014-11-28: \u5916\u8cc7\u8cb7\u8ce3\u8d85\u80a1\u6578, \u6295\u4fe1\u8cb7\u8ce3\u8d85\u80a1\u6578, \u81ea\u71df\u5546\u8cb7\u8ce3\u8d85\u80a1\u6578 (no own/hedging split)\n#   E2 2014-12-01 .. 2017-12-15: as E1 plus \u81ea\u71df\u5546\u8cb7\u8ce3\u8d85\u80a1\u6578(\u81ea\u884c\u8cb7\u8ce3) and \u81ea\u71df\u5546\u8cb7\u8ce3\u8d85\u80a1\u6578(\u907f\u96aa)\n#   E3 2017-12-18 ..           : \u5916\u9678\u8cc7\u8cb7\u8ce3\u8d85\u80a1\u6578(\u4e0d\u542b\u5916\u8cc7\u81ea\u71df\u5546) and \u5916\u8cc7\u81ea\u71df\u5546\u8cb7\u8ce3\u8d85\u80a1\u6578 replace \u5916\u8cc7\u8cb7\u8ce3\u8d85\u80a1\u6578\n# Each series is read from the first name in its list that the day's schema contains. Foreign is the reported foreign\n# total in E1-E2 and the sum of its two parts in E3 (spec: \"Foreign (including foreign dealers)\"). A series that a\n# day's schema does not report is NaN for every stock that day (never zero). An unknown schema stops the run.\nT86_FIELDS = {\n    \"foreign_ex\": [\"\u5916\u9678\u8cc7\u8cb7\u8ce3\u8d85\u80a1\u6578(\u4e0d\u542b\u5916\u8cc7\u81ea\u71df\u5546)\", \"\u5916\u8cc7\u8cb7\u8ce3\u8d85\u80a1\u6578\"],\n    \"foreign_dealer\": [\"\u5916\u8cc7\u81ea\u71df\u5546\u8cb7\u8ce3\u8d85\u80a1\u6578\", None],     # None: contained in the E1-E2 foreign total, read as 0\n    \"trust\": [\"\u6295\u4fe1\u8cb7\u8ce3\u8d85\u80a1\u6578\"],\n    \"dealer_total\": [\"\u81ea\u71df\u5546\u8cb7\u8ce3\u8d85\u80a1\u6578\"],\n    \"dealer_own\": [\"\u81ea\u71df\u5546\u8cb7\u8ce3\u8d85\u80a1\u6578(\u81ea\u884c\u8cb7\u8ce3)\"],\n    \"dealer_hedge\": [\"\u81ea\u71df\u5546\u8cb7\u8ce3\u8d85\u80a1\u6578(\u907f\u96aa)\"],\n    \"total3\": [\"\u4e09\u5927\u6cd5\u4eba\u8cb7\u8ce3\u8d85\u80a1\u6578\"],\n}\nREQUIRED = (\"foreign_ex\", \"trust\", \"dealer_total\")\nQUOTE_TITLE_SUFFIX = \"\u6b0a\u8b49\u3001\u725b\u718a\u8b49\u3001\u53ef\u5c55\u5ef6\u725b\u718a\u8b49))\"\nINDEX_NAME = \"\u767c\u884c\u91cf\u52a0\u6b0a\u80a1\u50f9\u6307\u6578\"\n\n\ndef _num(s):\n    s = str(s).replace(\",\", \"\").strip()\n    if s in (\"\", \"--\", \"---\", \"X\"):\n        return math.nan\n    try:\n        return float(s)\n    except ValueError:\n        return math.nan\n\n\ndef t86_positions(fields):\n    \"\"\"Column index per series for one schema, None if the series is contained elsewhere (read as 0), -1 if absent.\"\"\"\n    pos = {}\n    for k, names in T86_FIELDS.items():\n        hit = next((n for n in names if n is not None and n in fields), \"absent\")\n        if hit == \"absent\" and None in names and any(n in fields for n in T86_FIELDS[\"foreign_ex\"][1:]):\n            pos[k] = None\n        else:\n            pos[k] = fields.index(hit) if hit != \"absent\" else -1\n    for k in REQUIRED:\n        if pos[k] is None or pos[k] < 0:\n            raise ValueError(f\"unknown T86 schema: required series {k} not found in {fields}\")\n    return pos\n\n\ndef parse_t86(j):\n    \"\"\"Returns (records, unavailable): records[code][series] and the set of series this day's schema lacks.\"\"\"\n    pos = t86_positions(j[\"fields\"])\n    unavailable = {k for k, i in pos.items() if i is not None and i < 0}\n    out = {}\n    for row in j.get(\"data\", []):\n        code = row[0].strip()\n        if len(code) == 4 and code.isdigit():\n            out[code] = {k: (0.0 if i is None else (math.nan if i < 0 else _num(row[i]))) for k, i in pos.items()}\n    return out, unavailable\n\n\ndef t86_diagnostics(day_files):\n    \"\"\"Flow-only accounting checks per schema era (no prices, no forecasts). Shares of stock-days on which the\n    three-institution total equals the sum of the parts, with and without the foreign dealers, and on which the dealer\n    total equals own + hedging; and the foreign dealers' share of absolute foreign net flow.\"\"\"\n    eras = {}\n    for d, t86, _ in day_files:\n        key = tuple(t86[\"fields\"])\n        e = eras.setdefault(key, {\"first\": d, \"last\": d, \"days\": 0, \"rows\": 0, \"total_with_fd\": 0, \"total_without_fd\": 0,\n                                  \"dealer_rows\": 0, \"dealer_sum_ok\": 0, \"abs_fd\": 0.0, \"abs_foreign\": 0.0})\n        e[\"last\"] = d; e[\"days\"] += 1\n        rec, unavail = parse_t86(t86)\n        for r in rec.values():\n            vals = [r[\"foreign_ex\"], r[\"foreign_dealer\"], r[\"trust\"], r[\"dealer_total\"], r[\"total3\"]]\n            if not all(np.isfinite(v) for v in vals):\n                continue\n            e[\"rows\"] += 1\n            e[\"total_with_fd\"] += abs(r[\"total3\"] - (r[\"foreign_ex\"] + r[\"foreign_dealer\"] + r[\"trust\"] + r[\"dealer_total\"])) < 0.5\n            e[\"total_without_fd\"] += abs(r[\"total3\"] - (r[\"foreign_ex\"] + r[\"trust\"] + r[\"dealer_total\"])) < 0.5\n            if np.isfinite(r[\"dealer_own\"]) and np.isfinite(r[\"dealer_hedge\"]):\n                e[\"dealer_rows\"] += 1\n                e[\"dealer_sum_ok\"] += abs(r[\"dealer_total\"] - r[\"dealer_own\"] - r[\"dealer_hedge\"]) < 0.5\n            e[\"abs_fd\"] += abs(r[\"foreign_dealer\"]); e[\"abs_foreign\"] += abs(r[\"foreign_ex\"] + r[\"foreign_dealer\"])\n    out = []\n    for key, e in eras.items():\n        n = max(e[\"rows\"], 1)\n        out.append({\"first\": e[\"first\"], \"last\": e[\"last\"], \"days\": e[\"days\"], \"rows\": e[\"rows\"],\n                    \"share_total_eq_parts_with_foreign_dealers\": e[\"total_with_fd\"] / n,\n                    \"share_total_eq_parts_without_foreign_dealers\": e[\"total_without_fd\"] / n,\n                    \"share_dealer_total_eq_own_plus_hedge\": (e[\"dealer_sum_ok\"] / e[\"dealer_rows\"]) if e[\"dealer_rows\"] else None,\n                    \"foreign_dealer_share_of_abs_foreign_flow\": e[\"abs_fd\"] / e[\"abs_foreign\"] if e[\"abs_foreign\"] else None,\n                    \"fields\": list(key)})\n    return out\n\n\ndef parse_mi_index(j):\n    quotes, index = {}, math.nan\n    for t in j.get(\"tables\", []):\n        title = t.get(\"title\") or \"\"\n        fields = t.get(\"fields\") or []\n        if title.endswith(QUOTE_TITLE_SUFFIX) and \"\u8b49\u5238\u4ee3\u865f\" in fields:\n            ic, iv, ival = fields.index(\"\u8b49\u5238\u4ee3\u865f\"), fields.index(\"\u6210\u4ea4\u80a1\u6578\"), fields.index(\"\u6210\u4ea4\u91d1\u984d\")\n            icl, isg = fields.index(\"\u6536\u76e4\u50f9\"), fields.index(\"\u6f32\u8dcc(+/-)\")\n            for row in t.get(\"data\", []):\n                code = row[ic].strip()\n                if len(code) == 4 and code.isdigit():\n                    sign = re.sub(r\"<[^>]+>\", \"\", str(row[isg])).strip()\n                    quotes[code] = {\"vol\": _num(row[iv]), \"value\": _num(row[ival]), \"close\": _num(row[icl]),\n                                    \"event\": sign == \"X\"}\n        if fields and t.get(\"data\") and any(str(r[0]).strip() == INDEX_NAME for r in t[\"data\"]):\n            for r in t[\"data\"]:\n                if str(r[0]).strip() == INDEX_NAME:\n                    index = _num(r[1])\n    return quotes, index\n\n\ndef parse_twt49u(j):\n    \"\"\"Ex-rights / ex-dividend calculation table. Returns a set of (YYYYMMDD, code). Field names are matched by\n    content because probe 2 returned no rows; unparsed tables are reported, not guessed.\"\"\"\n    out = set()\n    f = j.get(\"fields\") or []\n    idate = next((i for i, x in enumerate(f) if \"\u65e5\u671f\" in x), None)\n    icode = next((i for i, x in enumerate(f) if \"\u4ee3\u865f\" in x), None)\n    if idate is None or icode is None:\n        return out\n    for row in j.get(\"data\", []):\n        digits = re.sub(r\"[^0-9]\", \"\", str(row[idate]))\n        if len(digits) == 7:\n            digits = str(int(digits[:3]) + 1911) + digits[3:]\n        code = str(row[icode]).strip()\n        if len(digits) == 8 and len(code) == 4 and code.isdigit():\n            out.add((digits, code))\n    return out\n\n\n@dataclass\nclass Panel:\n    dates: list\n    codes: list\n    vol: np.ndarray       # stocks x dates\n    value: np.ndarray\n    close: np.ndarray\n    event: np.ndarray     # bool\n    flows: dict           # class -> stocks x dates\n    index: np.ndarray     # dates\n\n\ndef build_panel(day_files, extra_events=None):\n    \"\"\"day_files: list of (date 'YYYYMMDD', t86_json, mi_json) in calendar order (trading days only).\n    extra_events: optional set of (YYYYMMDD, code) from TWT49U, united with the X change-sign flag.\"\"\"\n    parsed = []\n    codes = set()\n    for d, t86, mi in day_files:\n        fl, unavail = parse_t86(t86)\n        q, idx = parse_mi_index(mi)\n        parsed.append((d, (fl, unavail), q, idx))\n        codes |= set(q)\n    codes = sorted(codes)\n    ci = {c: i for i, c in enumerate(codes)}\n    N, T = len(codes), len(parsed)\n    nan = lambda: np.full((N, T), np.nan)\n    vol, value, close = nan(), nan(), nan()\n    event = np.zeros((N, T), bool)\n    flows = {k: nan() for k in T86_FIELDS}\n    index = np.full(T, np.nan)\n    for t, (d, (fl, unavail), q, idx) in enumerate(parsed):\n        index[t] = idx\n        for c, rec in q.items():\n            i = ci[c]\n            vol[i, t], value[i, t], close[i, t], event[i, t] = rec[\"vol\"], rec[\"value\"], rec[\"close\"], rec[\"event\"]\n            if rec[\"vol\"] and rec[\"vol\"] > 0:\n                f = fl.get(c)\n                for k in T86_FIELDS:\n                    # absent from the day's report = no institutional trade (0); a series the schema lacks = NaN\n                    flows[k][i, t] = math.nan if k in unavail else (0.0 if f is None else f[k])\n    x_only = int(event.sum())\n    if extra_events:\n        di = {p[0]: t for t, p in enumerate(parsed)}\n        for d, c in extra_events:\n            if d in di and c in ci:\n                event[ci[c], di[d]] = True\n    P = Panel([p[0] for p in parsed], codes, vol, value, close, event, flows, index)\n    P.event_counts = {\"x_flag\": x_only, \"union_with_twt49u\": int(event.sum())}\n    return P\n\n\ndef class_flows(P):\n    F = P.flows\n    out = {\n        \"Foreign\": F[\"foreign_ex\"] + F[\"foreign_dealer\"],\n        \"Trust\": F[\"trust\"],\n        \"DealerOwn\": F[\"dealer_own\"],\n        \"Hedge\": F[\"dealer_hedge\"],\n        \"DealerTotal\": F[\"dealer_total\"],   # amendment A2: the \"hedging included in the dealer class\" sensitivity\n    }\n    # Residual = minus all reported institutional flow. The dealer total replaces own + hedging, which it equals\n    # wherever the split is reported (checked by t86_diagnostics), and exists in every schema era.\n    out[\"Residual\"] = -(out[\"Foreign\"] + out[\"Trust\"] + F[\"dealer_total\"])\n    return out\n\n\ndef prepare(P, dev_mask, top_n=300, min_share=0.95, trail=60, post_listing=60):\n    \"\"\"Universe, flow scaling, beta, market-reference basis x, limit indicator and usable-origin masks.\"\"\"\n    N, T = P.close.shape\n    traded = np.isfinite(P.vol) & (P.vol > 0) & np.isfinite(P.close)\n    logc = np.log(P.close)\n    ret = np.full((N, T), np.nan)\n    ret[:, 1:] = logc[:, 1:] - logc[:, :-1]\n    prev_close = np.full((N, T), np.nan)\n    prev_close[:, 1:] = P.close[:, :-1]\n    limit = np.abs(P.close - prev_close) >= 0.095 * prev_close\n    limit &= np.isfinite(prev_close)\n    ri = np.full(T, np.nan)\n    ri[1:] = np.diff(np.log(P.index))\n    excl = P.event.copy()\n    excl[:, 1:] |= P.event[:, :-1]\n    # first `post_listing` traded days of a stock that starts trading after the first date\n    first = np.argmax(traded, axis=1)\n    for i in range(N):\n        if first[i] > 0:\n            excl[i, first[i]:first[i] + post_listing] = True\n    # universe on development days\n    dev = dev_mask[None, :]\n    usable_share = (traded & ~excl & dev).sum(1) / max(dev_mask.sum(), 1)\n    turnover = np.where(dev & traded, P.value, np.nan)\n    mean_turn = np.nanmean(turnover, axis=1)\n    elig = np.where((usable_share >= min_share) & np.isfinite(mean_turn))[0]\n    order = elig[np.lexsort((np.array(P.codes)[elig], -mean_turn[elig]))]\n    U = order[:top_n]\n    res = {\"universe_codes\": [P.codes[i] for i in U]}\n    # beta on development days\n    beta = np.full(N, np.nan)\n    for i in U:\n        m = dev_mask & traded[i] & ~excl[i] & ~limit[i] & np.isfinite(ret[i]) & np.isfinite(ri)\n        if m.sum() > 100:\n            X = ri[m]\n            beta[i] = float(np.dot(X - X.mean(), ret[i, m] - ret[i, m].mean()) / np.dot(X - X.mean(), X - X.mean()))\n    # adjusted log level: event-day return replaced by beta * index return; missing returns count as 0\n    adj = np.where(P.event, beta[:, None] * ri[None, :], ret)\n    adj = np.where(np.isfinite(adj), adj, 0.0)\n    level = np.cumsum(adj, axis=1)\n    logI = np.log(P.index)\n    basis = level - beta[:, None] * (logI[None, :] - logI[0])\n    x = np.full((N, T), np.nan)\n    for t in range(trail, T):\n        x[:, t] = basis[:, t] - basis[:, t - trail:t].mean(1)\n    flows = class_flows(P)\n    scale = np.nanmean(np.where(dev & traded, P.vol, np.nan), axis=1)\n    u = {k: v / scale[:, None] for k, v in flows.items()}\n    valid = traded & ~excl\n    usable = np.zeros((N, T), bool)\n    usable[:, 1:-1] = valid[:, :-2] & valid[:, 1:-1] & valid[:, 2:]\n    usable[:, 1:-1] &= np.isfinite(x[:, 1:-1]) & np.isfinite(x[:, :-2])\n    res.update({\"U\": U, \"x\": x, \"u\": u, \"limit\": limit.astype(float), \"usable\": usable, \"beta\": beta})\n    return res\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# 3. Run\n# ----------------------------------------------------------------------------------------------------------------\n\ndef run_period(prep, fit_mask, eval_mask, cls=\"Foreign\", min_share=0.95, B=9999, mean_block=20, seed=SEED):\n    U = prep[\"U\"]\n    rows = []\n    for i in U:\n        if prep[\"usable\"][i][eval_mask].mean() < min_share:\n            continue\n        ell, _ = stock_losses(prep[\"u\"][cls][i], prep[\"x\"][i], prep[\"limit\"][i], prep[\"usable\"][i], fit_mask, eval_mask)\n        if np.isfinite(ell).any():\n            rows.append(ell)\n    L = np.vstack(rows)\n    D, per = panel_D(L)\n    boot = bootstrap_D(L, mean_block=mean_block, B=B, seed=seed)\n    out = decide(D, boot)\n    out.update({\"class\": cls, \"n_stocks\": int(L.shape[0]), \"share_stocks_positive\": float((per > 0).mean()),\n                \"mean_block\": mean_block, \"B\": B})\n    return out\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# 4. Pre-specified diagnostics and secondary outcomes (amendment A2, P1T_amendment_A2.md)\n#    Nothing above this block is changed by A2 except the extra DealerTotal class. The primary statistic is\n#    computed only by run_period.\n# ----------------------------------------------------------------------------------------------------------------\n\nDIAG_B = 1999\n\n\ndef panel_L(prep, fit_mask, eval_mask, cls=\"Foreign\", x=None, usable=None, min_share=0.95):\n    \"\"\"Stocks x dates loss-gain matrix, as in run_period, with optional replacement basis x and usable mask.\"\"\"\n    X = prep[\"x\"] if x is None else x\n    US = prep[\"usable\"] if usable is None else usable\n    rows = []\n    for i in prep[\"U\"]:\n        if US[i][eval_mask].mean() < min_share:\n            continue\n        ell, _ = stock_losses(prep[\"u\"][cls][i], X[i], prep[\"limit\"][i], US[i], fit_mask, eval_mask)\n        if np.isfinite(ell).any():\n            rows.append(ell)\n    return np.vstack(rows) if rows else np.zeros((0, len(eval_mask)))\n\n\ndef fast_stationary_indices(n, mean_block, rng):\n    \"\"\"Vectorized stationary-bootstrap indices (same law as stationary_indices, different random stream).\"\"\"\n    restart = rng.random(n) < 1.0 / mean_block\n    restart[0] = True\n    starts = rng.integers(n, size=n)\n    pos = np.arange(n)\n    last = np.maximum.accumulate(np.where(restart, pos, 0))\n    return (starts[last] + (pos - last)) % n\n\n\ndef bootstrap_D_fast(L, mean_block=20, B=DIAG_B, seed=SEED):\n    cols = np.where(np.isfinite(L).any(axis=0))[0]\n    Lc = L[:, cols]; rng = np.random.default_rng(seed); n = Lc.shape[1]\n    S = np.nan_to_num(Lc); C = np.isfinite(Lc).astype(float)\n    out = np.empty(B)\n    for b in range(B):\n        idx = fast_stationary_indices(n, mean_block, rng)\n        num = S[:, idx].sum(1); cnt = C[:, idx].sum(1)\n        out[b] = (num[cnt > 0] / cnt[cnt > 0]).mean()\n    return out\n\n\ndef summarize_L(L, mean_block=20, B=DIAG_B, seed=SEED):\n    if L.shape[0] == 0 or not np.isfinite(L).any():\n        return {\"D\": None, \"verdict\": \"not computable\", \"n_stocks\": int(L.shape[0])}\n    D, per = panel_D(L)\n    out = decide(D, bootstrap_D_fast(L, mean_block=mean_block, B=B, seed=seed))\n    out.update({\"n_stocks\": int(L.shape[0]), \"share_stocks_positive\": float((per > 0).mean()), \"B\": B,\n                \"mean_block\": mean_block})\n    return out\n\n\ndef residual_basis(prep, fit_mask, classes=(\"Foreign\", \"Trust\", \"DealerOwn\", \"Hedge\"), max_zero_share=0.5):\n    \"\"\"Spec 7.1. Per stock, OLS of x_t on a constant, x_{t-1} and the reported flows u_{c,t}, u_{c,t-1} of the\n    classes and of dealer hedging, fitted on fit-period days where all are finite; returns x_t minus the fitted value\n    (NaN where a regressor is missing). Under the confound (prices reveal past flow) the residual carries no\n    information about future flow; under feedback it keeps the price news that flows respond to.\n\n    Amendment A3 (validation finding, approved before the holdout):\n    - The Residual class is not a regressor. It equals minus the sum of the other reported flows wherever the dealer\n      split is reported, so it adds no column space; with it the design is exactly singular.\n    - A class whose flow is exactly zero on more than `max_zero_share` of the stock's fit-period days is not a\n      regressor for that stock: its coefficient would be estimated from a handful of days and extrapolated to an\n      evaluation period in which the class may be active (dealer hedging, 2014-16 vs 2017-18).\"\"\"\n    x = prep[\"x\"]; N, T = x.shape\n    xr = np.full((N, T), np.nan)\n    for i in prep[\"U\"]:\n        cols = [np.ones(T), np.r_[np.nan, x[i, :-1]]]\n        base_ok = np.isfinite(x[i]) & np.isfinite(cols[1]) & fit_mask\n        for c in classes:\n            u = prep[\"u\"][c][i]\n            fin = base_ok & np.isfinite(u)\n            if fin.sum() == 0 or np.mean(u[fin] == 0) > max_zero_share:\n                continue\n            cols += [u, np.r_[np.nan, u[:-1]]]\n        Xm = np.column_stack(cols)\n        ok = np.isfinite(Xm).all(1) & np.isfinite(x[i])\n        fit = ok & fit_mask\n        if fit.sum() < 50:\n            continue\n        b = np.linalg.lstsq(Xm[fit], x[i, fit], rcond=None)[0]\n        xr[i, ok] = x[i, ok] - Xm[ok] @ b\n    us = prep[\"usable\"].copy()\n    us[:, 1:] &= np.isfinite(xr[:, 1:]) & np.isfinite(xr[:, :-1])\n    us[:, 0] = False\n    return xr, us\n\n\ndef limit_free_usable(prep):\n    \"\"\"Spec 7.4: limit days and the following day dropped. An origin t is dropped if t-1, t or t+1 is a limit day or\n    the day after one.\"\"\"\n    lim = prep[\"limit\"].astype(bool)\n    bad = lim.copy(); bad[:, 1:] |= lim[:, :-1]\n    us = prep[\"usable\"].copy()\n    us[:, 1:-1] &= ~bad[:, :-2] & ~bad[:, 1:-1] & ~bad[:, 2:]\n    return us\n\n\ndef aggregate_k(prep, k, fit_mask, eval_mask, cls=\"Foreign\"):\n    \"\"\"Spec 7.4: k-day periods. Non-overlapping blocks of k trading days from the first date; flow = sum over the\n    block, basis = x on the block's last day, limit = any limit day in the block. A block is valid if every day is a\n    finite in u (missing or excluded days are NaN) and x is finite on the block's last day. A stock enters if at least\n    80% of its evaluation blocks are usable. Fit and evaluation blocks are those whose last day is in the mask.\"\"\"\n    u_all = prep[\"u\"][cls]; x = prep[\"x\"]; lim = prep[\"limit\"]\n    N, T = x.shape; nb = T // k\n    last = np.arange(nb) * k + (k - 1)\n    fit_b, ev_b = fit_mask[last], eval_mask[last]\n    rows = []\n    for i in prep[\"U\"]:\n        ub = u_all[i, :nb * k].reshape(nb, k).sum(1)               # NaN if any day missing\n        xb = x[i, last]\n        lb = lim[i, :nb * k].reshape(nb, k).max(1)\n        val = np.isfinite(ub) & np.isfinite(xb)\n        usb = np.zeros(nb, bool)\n        usb[1:-1] = val[:-2] & val[1:-1] & val[2:]\n        if usb[ev_b].mean() < 0.80:   # a block is lost if any of its days is; threshold 80% of evaluation blocks\n            continue\n        ell, _ = stock_losses(ub, xb, lb, usb, fit_b, ev_b)\n        if np.isfinite(ell).any():\n            rows.append(ell)\n    L = np.vstack(rows) if rows else np.zeros((0, nb))\n    return L, max(2, int(round(20 / k)))\n\n\ndef iv_tagged(u, x, ok, hac_lag=None):\n    \"\"\"Proposition 6.1 tagged IV for one stock: u_t = c + r u_{t-1} + w x_t + e_t, instruments (1, x_{t-1}, x_t).\n    ok[t] marks usable equations. Returns beta = (c, r, w), Newey-West HAC covariance, residuals and T.\"\"\"\n    t = np.where(ok)[0]\n    t = t[t >= 1]\n    if len(t) < 60:\n        return None\n    y = u[t]; X = np.column_stack([np.ones(len(t)), u[t - 1], x[t]]); Z = np.column_stack([np.ones(len(t)), x[t - 1], x[t]])\n    ZX = Z.T @ X\n    if abs(np.linalg.det(ZX)) < 1e-12:\n        return None\n    b = np.linalg.solve(ZX, Z.T @ y)\n    e = y - X @ b\n    n = len(t)\n    L = hac_lag if hac_lag is not None else int(np.floor(4 * (n / 100) ** (2 / 9)))\n    g = Z * e[:, None]\n    S = g.T @ g\n    for l in range(1, L + 1):\n        G = g[l:].T @ g[:-l]\n        S += (1 - l / (L + 1)) * (G + G.T)\n    Ainv = np.linalg.inv(ZX)\n    V = Ainv @ S @ Ainv.T\n    return {\"beta\": b, \"V\": V, \"resid\": e, \"T\": n, \"rows\": t}\n\n\ndef _iv_ok(prep, i, cls, mask):\n    u = prep[\"u\"][cls][i]; x = prep[\"x\"][i]\n    ok = prep[\"usable\"][i] & mask & np.isfinite(u)\n    ok[1:] &= np.isfinite(u[:-1]) & np.isfinite(x[:-1])\n    ok[0] = False\n    return ok\n\n\ndef secondary_S2(prep, fit_mask, eval_mask, classes=(\"Foreign\", \"Trust\", \"DealerOwn\")):\n    \"\"\"Spec 8, S2: tagged IV estimates (r_i, w_i) per class with HAC 95% intervals, in the fit and evaluation periods;\n    for Foreign, the share of stocks whose evaluation-period r_F lies in its fit-period 95% interval.\"\"\"\n    out = {}\n    for c in classes:\n        rec = {\"fit\": [], \"eval\": []}\n        inside = []\n        for i in prep[\"U\"]:\n            r = {}\n            for name, m in ((\"fit\", fit_mask), (\"eval\", eval_mask)):\n                est = iv_tagged(prep[\"u\"][c][i], prep[\"x\"][i], _iv_ok(prep, i, c, m))\n                if est is not None:\n                    se = np.sqrt(np.diag(est[\"V\"]))\n                    r[name] = (est[\"beta\"][1], se[1], est[\"beta\"][2], se[2])\n                    rec[name].append(r[name])\n            if c == \"Foreign\" and \"fit\" in r and \"eval\" in r:\n                rf, sf = r[\"fit\"][0], r[\"fit\"][1]\n                inside.append(abs(r[\"eval\"][0] - rf) <= 1.96 * sf)\n        summ = {}\n        for name in (\"fit\", \"eval\"):\n            a = np.array(rec[name]) if rec[name] else np.zeros((0, 4))\n            if len(a):\n                summ[name] = {\"n_stocks\": int(len(a)), \"r_median\": float(np.median(a[:, 0])),\n                              \"r_iqr\": [float(np.percentile(a[:, 0], 25)), float(np.percentile(a[:, 0], 75))],\n                              \"w_median\": float(np.median(a[:, 2])),\n                              \"w_iqr\": [float(np.percentile(a[:, 2], 25)), float(np.percentile(a[:, 2], 75))],\n                              \"share_w_positive_significant\": float(np.mean(a[:, 2] - 1.96 * a[:, 3] > 0)),\n                              \"share_w_negative_significant\": float(np.mean(a[:, 2] + 1.96 * a[:, 3] < 0))}\n        if c == \"Foreign\":\n            summ[\"share_eval_rF_in_fit_interval\"] = float(np.mean(inside)) if inside else None\n        out[c] = summ\n    return out\n\n\ndef formula_se_r(beta, resid, x, ok_fit, T_eval, S=200):\n    \"\"\"Eq. (ivvar) evaluated with fit-period estimates: gamma_x from the fit-period basis (demeaned), r, w from the\n    fit-period IV, sigma_f^2 = Var(e)/(1 + r^2). Returns the predicted standard error of r_hat for T_eval equations.\"\"\"\n    r, w = beta[1], beta[2]\n    xs = x[ok_fit]; xs = xs - xs.mean(); n = len(xs)\n    if n < 2 * S:\n        S = n // 4\n    gam = np.array([np.dot(xs[:n - s], xs[s:]) / n for s in range(S + 2)])\n    A = lambda m: sum((r ** s) * gam[s + m] for s in range(S + 1 - m))\n    Sig = np.array([[w * A(0), gam[1]], [w * A(1), gam[0]]])\n    sf2 = np.var(resid) / (1 + r * r)\n    G0 = np.array([[gam[0], gam[1]], [gam[1], gam[0]]])\n    G1 = np.array([[gam[1], gam[0]], [gam[2], gam[1]]])\n    Om = sf2 * ((1 + r * r) * G0 - r * (G1 + G1.T))\n    try:\n        Si = np.linalg.inv(Sig)\n    except np.linalg.LinAlgError:\n        return None\n    V = Si @ Om @ Si.T\n    return float(np.sqrt(max(V[0, 0], 0) / T_eval))\n\n\ndef secondary_S3(prep, fit_mask, eval_mask, B=199, mean_block=20, seed=SEED):\n    \"\"\"Spec 8, S3: for each stock, the eq. (ivvar) prediction of the evaluation-period standard error of r_F from\n    fit-period estimates, against a stationary block bootstrap standard error of r_F over evaluation equations.\n    Outcome: the median ratio across stocks, with agreement if it lies in [0.67, 1.5].\"\"\"\n    rng = np.random.default_rng(seed)\n    ratios = []\n    n_nonstat = 0\n    for i in prep[\"U\"]:\n        okf = _iv_ok(prep, i, \"Foreign\", fit_mask); oke = _iv_ok(prep, i, \"Foreign\", eval_mask)\n        f = iv_tagged(prep[\"u\"][\"Foreign\"][i], prep[\"x\"][i], okf)\n        e = iv_tagged(prep[\"u\"][\"Foreign\"][i], prep[\"x\"][i], oke)\n        if f is None or e is None:\n            continue\n        if not abs(f[\"beta\"][1]) < 1:          # amendment A3: eq. (ivvar) is defined only for |r| < 1\n            n_nonstat += 1\n            continue\n        pred = formula_se_r(f[\"beta\"], f[\"resid\"], prep[\"x\"][i], okf, e[\"T\"])\n        if pred is None or not np.isfinite(pred) or pred <= 0:\n            continue\n        t = e[\"rows\"]; u = prep[\"u\"][\"Foreign\"][i]; x = prep[\"x\"][i]\n        y = u[t]; X = np.column_stack([np.ones(len(t)), u[t - 1], x[t]]); Z = np.column_stack([np.ones(len(t)), x[t - 1], x[t]])\n        bs = []\n        for _ in range(B):\n            idx = fast_stationary_indices(len(t), mean_block, rng)\n            ZX = Z[idx].T @ X[idx]\n            if abs(np.linalg.det(ZX)) < 1e-12:\n                continue\n            bs.append(np.linalg.solve(ZX, Z[idx].T @ y[idx])[1])\n        if len(bs) < B // 2:\n            continue\n        ratios.append(pred / float(np.std(bs)))\n    ratios = np.array(ratios)\n    if not len(ratios):\n        return {\"n_stocks\": 0, \"verdict\": \"not computable\", \"n_excluded_abs_r_ge_1\": n_nonstat}\n    med = float(np.median(ratios))\n    return {\"n_stocks\": int(len(ratios)), \"median_ratio_formula_to_bootstrap\": med,\n            \"iqr_ratio\": [float(np.percentile(ratios, 25)), float(np.percentile(ratios, 75))],\n            \"share_stocks_in_range\": float(np.mean((ratios >= 0.67) & (ratios <= 1.5))),\n            \"agreement\": bool(0.67 <= med <= 1.5), \"B_per_stock\": B, \"n_excluded_abs_r_ge_1\": n_nonstat}\n\n\ndef holm(pvals):\n    \"\"\"Holm step-down adjusted p-values for a dict name -> p.\"\"\"\n    items = sorted(pvals.items(), key=lambda kv: kv[1]); m = len(items); out = {}; run = 0.0\n    for j, (k, p) in enumerate(items):\n        run = max(run, min(1.0, (m - j) * p)); out[k] = run\n    return out\n\n\ndef diagnostics(prep, dates, fit_mask, eval_mask, B=DIAG_B, seed=SEED):\n    \"\"\"All pre-specified diagnostics (spec 7) and secondary outcomes (spec 8) for one period. dates: 'YYYYMMDD'.\"\"\"\n    dates = np.array(dates)\n    res = {\"B\": B}\n    # 7.1 reverse-causality check\n    xr, usr = residual_basis(prep, fit_mask)\n    Lr = panel_L(prep, fit_mask, eval_mask, \"Foreign\", x=xr, usable=usr)\n    res[\"reverse_causality\"] = summarize_L(Lr, B=B, seed=seed)\n    if Lr.shape[0]:                            # amendment A3: descriptive only; the verdict stays on D\n        res[\"reverse_causality\"][\"median_d_descriptive\"] = float(np.median(np.nanmean(Lr, axis=1)))\n    # 7.2 per-stock share (with the primary L)\n    L = panel_L(prep, fit_mask, eval_mask, \"Foreign\")\n    per = np.nanmean(L, axis=1)\n    res[\"per_stock\"] = {\"share_positive\": float(np.mean(per > 0)), \"median_d\": float(np.median(per)),\n                        \"quartiles_d\": [float(np.percentile(per, 25)), float(np.percentile(per, 75))]}\n    # 7.3 stability by calendar year, and Feb-Jun 2020 separately\n    years = sorted({d[:4] for d in dates[eval_mask]})\n    subs = {y: np.array([d[:4] == y for d in dates]) & eval_mask for y in years}\n    subs[\"2020-02_to_2020-06\"] = np.array([\"20200201\" <= d <= \"20200630\" for d in dates]) & eval_mask\n    stab = {}\n    for name, m in subs.items():\n        if m.sum() < 20:\n            continue\n        Ls = np.where(m[None, :], L, np.nan)\n        keep = np.isfinite(Ls).sum(1) >= 10\n        stab[name] = summarize_L(Ls[keep], B=B, seed=seed)\n    res[\"stability\"] = stab\n    # 7.4 sensitivity\n    sens = {}\n    sens[\"hedging_in_dealer_class\"] = summarize_L(panel_L(prep, fit_mask, eval_mask, \"DealerTotal\"), B=B, seed=seed)\n    sens[\"net_value_or_market_cap_scaling\"] = {\"verdict\": \"not computable\",\n                                               \"reason\": \"shares outstanding are not in the registered sources\"}\n    sens[\"limit_days_dropped\"] = summarize_L(panel_L(prep, fit_mask, eval_mask, \"Foreign\", usable=limit_free_usable(prep)), B=B, seed=seed)\n    for k in (2, 5):\n        Lk, mb = aggregate_k(prep, k, fit_mask, eval_mask)\n        sens[f\"{k}_day_periods\"] = summarize_L(Lk, mean_block=mb, B=B, seed=seed)\n    res[\"sensitivity\"] = sens\n    # 8 secondary outcomes\n    S1 = {}\n    for c in (\"Trust\", \"DealerOwn\"):\n        Lc = panel_L(prep, fit_mask, eval_mask, c)\n        S1[c] = run_period(prep, fit_mask, eval_mask, c) if Lc.shape[0] else {\"D\": None, \"verdict\": \"not computable\",\n                                                                              \"p_one_sided\": 1.0, \"n_stocks\": 0}\n    pv = {c: S1[c][\"p_one_sided\"] for c in S1}\n    res[\"S1\"] = {\"by_class\": S1, \"holm_adjusted_p\": holm(pv)}\n    res[\"S2\"] = secondary_S2(prep, fit_mask, eval_mask)\n    res[\"S3\"] = secondary_S3(prep, fit_mask, eval_mask)\n    return res\n\n\nif __name__ == \"__main__\":\n    print(\"P1T executor. Use the Colab notebook or validate_executor.py.\")\n"
CORE_SRC = "\"\"\"P1M: the identification barrier measured in a real market (protocol P1M_measurement_protocol.md).\n\nTaiwan development period 2013-01-02 to 2016-12-30 only. Input: the `prep` dictionary of p1t_executor.prepare (universe,\nmarket-reference basis x, class flows u scaled by development-period average volume, usable-origin mask).\n\nLayers\n------\n1. Estimation: per-class feedback rules u_{c,t} = r_c u_{c,t-1} + w_c x_{t-d} (stock fixed effects, pooled OLS) and the\n   price equation x_t = sum_j a_j x_{t-j} + kappa U_{t-1} (U = sum of the class flows), with a stationary date-block\n   bootstrap that resamples the same dates for every equation.\n2. The price-only experiment: A(z) = 1 - sum_j a_j z^j - kappa z^(1+d) sum_c w_c / (1 - r_c z). With the market's\n   impact function and the non-feedback order flow held at their estimates, the spectral density of x is g / |A|^2 for\n   a fixed g, so the Whittle divergence rate between two compositions depends on A alone.\n3. The three outcomes: Q1 years to separate the classes; Q2 range of the foreign-halt effect, and Q3 range of the\n   uniform-cut effect, over compositions the whole development record cannot tell apart.\n\"\"\"\nfrom __future__ import annotations\n\nimport itertools\nimport math\n\nimport numpy as np\n\nSEED = 20261121\nB = 999\nMEAN_BLOCK = 20\nP_LAGS = 5\nDELAY = 1\nJ0 = 0.02            # no test of any size has power above size + sqrt(J0/2) = size + 0.10 (Pinsker)\nPOWER, SIZE = 0.80, 0.05\nM_GRID = 8192\nR_MAX = 0.99\nN_RANDOM_STARTS = 40\nCLASSES = (\"Foreign\", \"Trust\", \"Dealers\")\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# 1. Estimation\n# ----------------------------------------------------------------------------------------------------------------\n\ndef lag(a, k):\n    out = np.full(a.shape, np.nan)\n    if k == 0:\n        return a.copy()\n    out[:, k:] = a[:, :-k]\n    return out\n\n\ndef fe_moments(y, Xs, ok, min_rows=50):\n    \"\"\"Within-stock demeaned pooled regression of y on the columns Xs over rows ok. Returns per-date moment sums so\n    that a date bootstrap only re-sums them.\"\"\"\n    X = np.stack(Xs, -1)\n    ok = ok & np.isfinite(y) & np.isfinite(X).all(-1)\n    cnt = ok.sum(1)\n    ok &= (cnt >= min_rows)[:, None]\n    cnt = ok.sum(1)\n    c = np.maximum(cnt, 1)\n    ym = np.where(ok, y, 0).sum(1) / c\n    Xm = np.where(ok[..., None], X, 0).sum(1) / c[:, None]\n    yd = np.where(ok, y - ym[:, None], 0.0)\n    Xd = np.where(ok[..., None], X - Xm[:, None, :], 0.0)\n    return {\"M\": np.einsum(\"itk,itl->tkl\", Xd, Xd), \"m\": np.einsum(\"itk,it->tk\", Xd, yd), \"yy\": (yd ** 2).sum(0),\n            \"n\": ok.sum(0), \"n_stocks\": int((cnt > 0).sum()), \"ok\": ok, \"yd\": yd, \"Xd\": Xd}\n\n\ndef ols_from(mom, idx=None):\n    M = mom[\"M\"].sum(0) if idx is None else mom[\"M\"][idx].sum(0)\n    m = mom[\"m\"].sum(0) if idx is None else mom[\"m\"][idx].sum(0)\n    yy = mom[\"yy\"].sum() if idx is None else mom[\"yy\"][idx].sum()\n    n = mom[\"n\"].sum() if idx is None else mom[\"n\"][idx].sum()\n    b = np.linalg.solve(M, m)\n    ssr = yy - 2 * b @ m + b @ M @ b\n    dof = max(n - len(b) - mom[\"n_stocks\"], 1)\n    return b, float(ssr / dof), float(1 - ssr / yy) if yy > 0 else float(\"nan\")\n\n\ndef stationary_indices(n, mean_block, rng):\n    restart = rng.random(n) < 1.0 / mean_block\n    restart[0] = True\n    starts = rng.integers(n, size=n)\n    pos = np.arange(n)\n    last = np.maximum.accumulate(np.where(restart, pos, 0))\n    return (starts[last] + (pos - last)) % n\n\n\ndef class_flows(prep, dealer=\"DealerTotal\"):\n    u = prep[\"u\"]\n    return {\"Foreign\": u[\"Foreign\"], \"Trust\": u[\"Trust\"], \"Dealers\": u[dealer]}\n\n\ndef universe_mask(prep, T):\n    m = np.zeros((prep[\"x\"].shape[0], T), bool)\n    m[prep[\"U\"]] = True\n    return m\n\n\ndef build_equations(prep, mask, delay=DELAY, p=P_LAGS, dealer=\"DealerTotal\"):\n    \"\"\"mask: dates (bool, length T) to use. Returns the moment objects for the three rule equations, the price\n    equation, and the class-specific-impact diagnostic.\"\"\"\n    x = prep[\"x\"]\n    N, T = x.shape\n    base = prep[\"usable\"] & universe_mask(prep, T) & mask[None, :]\n    u = class_flows(prep, dealer)\n    eqs = {}\n    sig = lag(x, delay)\n    for c in CLASSES:\n        eqs[c] = fe_moments(u[c], [lag(u[c], 1), sig], base)\n    Usum = u[\"Foreign\"] + u[\"Trust\"] + u[\"Dealers\"]\n    xl = [lag(x, j) for j in range(1, p + 1)]\n    eqs[\"price\"] = fe_moments(x, xl + [lag(Usum, 1)], base)\n    eqs[\"price_by_class\"] = fe_moments(x, xl + [lag(u[c], 1) for c in CLASSES], base)\n    return eqs\n\n\ndef theta_from(betas, p=P_LAGS, delay=DELAY):\n    bp = betas[\"price\"]\n    return {\"a\": np.array(bp[:p]), \"kappa\": float(bp[p]), \"r\": np.array([betas[c][0] for c in CLASSES]),\n            \"w\": np.array([betas[c][1] for c in CLASSES]), \"delay\": delay}\n\n\ndef estimate(eqs, p=P_LAGS, delay=DELAY, B_=B, mean_block=MEAN_BLOCK, seed=SEED):\n    \"\"\"Point estimates and the joint stationary date bootstrap of (theta, H_c, U, S).\"\"\"\n    point = {}\n    stats = {}\n    for k in list(CLASSES) + [\"price\", \"price_by_class\"]:\n        b, s2, r2 = ols_from(eqs[k])\n        point[k] = b\n        stats[k] = {\"coef\": b.tolist(), \"sigma2\": s2, \"R2\": r2, \"rows\": int(eqs[k][\"n\"].sum()),\n                    \"stocks\": eqs[k][\"n_stocks\"]}\n    th = theta_from(point, p, delay)\n    T = len(eqs[\"price\"][\"n\"])\n    rng = np.random.default_rng(seed)\n    draws = []\n    for _ in range(B_):\n        idx = stationary_indices(T, mean_block, rng)\n        try:\n            bb = {k: ols_from(eqs[k], idx)[0] for k in list(CLASSES) + [\"price\"]}\n        except np.linalg.LinAlgError:\n            continue\n        t = theta_from(bb, p, delay)\n        H = effects(t)\n        draws.append(np.r_[t[\"r\"], t[\"w\"], t[\"kappa\"], t[\"a\"].sum(), H[\"H\"], H[\"U\"], H[\"S\"]])\n    draws = np.array(draws)\n    names = [f\"r_{c}\" for c in CLASSES] + [f\"w_{c}\" for c in CLASSES] + [\"kappa\", \"sum_a\"] + \\\n            [f\"H_{c}\" for c in CLASSES] + [\"U\", \"S\"]\n    pt = effects(th)\n    ptv = np.r_[th[\"r\"], th[\"w\"], th[\"kappa\"], th[\"a\"].sum(), pt[\"H\"], pt[\"U\"], pt[\"S\"]]\n    ci = {n: {\"est\": float(v), \"ci95\": [float(np.percentile(draws[:, j], 2.5)), float(np.percentile(draws[:, j], 97.5))],\n              \"se\": float(np.std(draws[:, j]))} for j, (n, v) in enumerate(zip(names, ptv))}\n    return th, stats, ci, len(draws)\n\n\ndef effects(th):\n    \"\"\"H_c = kappa w_c / (1 - r_c): class c's contribution to the feedback budget, i.e. the fall in S when class c's\n    feedback is halted. U = sum_c H_c: the fall in S when all feedback is cut (a uniform cut by a fraction l lowers S by\n    l U). S = sum_j a_j + U.\"\"\"\n    H = th[\"kappa\"] * th[\"w\"] / (1 - th[\"r\"])\n    return {\"H\": H, \"U\": float(H.sum()), \"S\": float(th[\"a\"].sum() + H.sum())}\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# 2. The price-only experiment\n# ----------------------------------------------------------------------------------------------------------------\n\n_OMEGA = 2 * np.pi * (np.arange(M_GRID) + 0.5) / M_GRID - np.pi\n_Z = np.exp(-1j * _OMEGA)\n\n\ndef A_of(th, r=None, w=None, z=_Z):\n    r = th[\"r\"] if r is None else np.asarray(r)\n    w = th[\"w\"] if w is None else np.asarray(w)\n    poly = 1 - sum(a * z ** (j + 1) for j, a in enumerate(th[\"a\"]))\n    K = sum(wc / (1 - rc * z) for rc, wc in zip(r, w))\n    return poly - th[\"kappa\"] * z ** (1 + th[\"delay\"]) * K\n\n\ndef kl_rate(th, r2, w2, A1=None):\n    \"\"\"Whittle divergence rate per observation, KL(P_theta || P_theta'), theta' = theta with rules (r2, w2):\n    (1/4 pi) int [f/f' - 1 - log(f/f')] d omega, with f/f' = |A'|^2 / |A|^2.\"\"\"\n    A1 = A_of(th) if A1 is None else A1\n    R = np.abs(A_of(th, r2, w2)) ** 2 / np.abs(A1) ** 2\n    return float(0.5 * np.mean(R - 1 - np.log(R)))\n\n\ndef char_poly(th, r=None, w=None):\n    \"\"\"Coefficients (increasing powers) of the polynomial whose roots are the poles of the price process:\n    (1 - sum a_j z^j) prod (1 - r_c z) - kappa z^(1+d) sum_c w_c prod_{e != c} (1 - r_e z).\"\"\"\n    from numpy.polynomial import polynomial as Pl\n    r = th[\"r\"] if r is None else np.asarray(r)\n    w = th[\"w\"] if w is None else np.asarray(w)\n    left = np.r_[1.0, -np.asarray(th[\"a\"])]\n    prod = np.array([1.0])\n    for rc in r:\n        prod = Pl.polymul(prod, [1.0, -rc])\n    left = Pl.polymul(left, prod)\n    right = np.array([0.0])\n    for c, wc in enumerate(w):\n        pc = np.array([1.0])\n        for e, re_ in enumerate(r):\n            if e != c:\n                pc = Pl.polymul(pc, [1.0, -re_])\n        right = Pl.polyadd(right, wc * pc)\n    shift = np.zeros(2 + th[\"delay\"]); shift[-1] = th[\"kappa\"]\n    right = Pl.polymul(shift, right)\n    return Pl.polysub(left, right)\n\n\ndef stationary(th, r=None, w=None, margin=1e-6):\n    roots = np.roots(char_poly(th, r, w)[::-1])\n    return bool(len(roots) == 0 or np.min(np.abs(roots)) > 1 + margin), float(np.min(np.abs(roots))) if len(roots) else float(\"inf\")\n\n\ndef _min_over_W(fun, lo=-50.0, hi=50.0):\n    from scipy.optimize import minimize_scalar\n    r = minimize_scalar(fun, bounds=(lo, hi), method=\"bounded\", options={\"xatol\": 1e-12})\n    return r.x, r.fun\n\n\ndef q1_price(th):\n    \"\"\"Smallest Whittle rate from the estimated population to any single-memory population (aggregate kernel\n    W R^m), over R in [-R_MAX, R_MAX] and W.\"\"\"\n    from scipy.optimize import minimize\n    A1 = A_of(th)\n    Uhat = effects(th)[\"U\"]\n    scale = max(abs(th[\"kappa\"]), 1e-12)\n    best = (np.inf, None, None)\n    for R in np.linspace(-R_MAX, R_MAX, 199):\n        Wk, f = _min_over_W(lambda W: kl_rate(th, [R, 0, 0], [W / scale, 0, 0], A1),\n                            lo=-10 * (abs(Uhat) + 1e-3), hi=10 * (abs(Uhat) + 1e-3))\n        if f < best[0]:\n            best = (f, R, Wk / scale)\n    f0, R0, W0 = best\n    g = lambda v: kl_rate(th, [np.clip(v[0], -R_MAX, R_MAX), 0, 0], [v[1], 0, 0], A1)\n    res = minimize(g, [R0, W0], method=\"Nelder-Mead\", options={\"xatol\": 1e-12, \"fatol\": 1e-20, \"maxiter\": 4000})\n    if res.fun < f0:\n        f0, R0, W0 = res.fun, float(np.clip(res.x[0], -R_MAX, R_MAX)), float(res.x[1])\n    return {\"rate\": float(f0), \"R\": float(R0), \"W\": float(W0)}\n\n\ndef pinsker_days(rate, power=POWER, size=SIZE):\n    \"\"\"Observations below which no test of size `size` has power `power`: KL >= 2 (power - size)^2 is necessary.\"\"\"\n    return 2 * (power - size) ** 2 / rate if rate > 0 else float(\"inf\")\n\n\ndef tag_filters(prep, mask, th, dealer=\"DealerTotal\", burn=100):\n    \"\"\"Pooled, within-stock demeaned series used by the tagged-record divergence: s_t = x_{t-d}, and the rows where\n    all three rule equations are usable. Returns (S, ok) as stocks x dates arrays.\"\"\"\n    x = prep[\"x\"]\n    N, T = x.shape\n    s = lag(x, th[\"delay\"])\n    u = class_flows(prep, dealer)\n    ok = prep[\"usable\"] & universe_mask(prep, T) & mask[None, :] & np.isfinite(s)\n    for c in CLASSES:\n        ok &= np.isfinite(u[c]) & np.isfinite(lag(u[c], 1))\n    sd = np.where(np.isfinite(s), s, np.nan)\n    mu = np.nanmean(np.where(ok, sd, np.nan), axis=1)\n    sd = sd - mu[:, None]\n    # rows need `burn` finite days of s before them, so the filters below have converged\n    fin = np.isfinite(sd)\n    run = np.zeros((N, T), int)\n    for t in range(T):\n        run[:, t] = np.where(fin[:, t], (run[:, t - 1] + 1) if t else 1, 0)\n    ok &= run > burn\n    return np.where(fin, sd, 0.0), ok\n\n\ndef ar_filter(s, R):\n    G = np.zeros_like(s)\n    for t in range(1, s.shape[1]):\n        G[:, t] = R * G[:, t - 1] + s[:, t]\n    return G\n\n\ndef q1_tags(th, S, ok, resid_cov):\n    \"\"\"Divergence rate per stock-day of the tagged record (prices and the three class flows) between the estimated\n    population and the closest single-memory population (decay R common to all classes, weights W_c free). With the\n    price equation the same under both, the rate is (1/2) E[d' Omega^{-1} d], d_c = w_c s_t - W_c q_c(R)_t,\n    q_c = (1 - r_c L)/(1 - R L) s.\"\"\"\n    Oi = np.linalg.inv(resid_cov)\n    w, r = th[\"w\"], th[\"r\"]\n    n = ok.sum()\n    s = S[ok]\n    S0 = float(np.mean(s * s))\n\n    def at(R):\n        G = ar_filter(S, R)\n        Gl = lag(G, 1)\n        q = [np.where(ok, G - rc * np.nan_to_num(Gl), 0)[ok] for rc in r]\n        Q = np.array([[np.mean(q[c] * q[e]) for e in range(3)] for c in range(3)])\n        Cq = np.array([np.mean(q[c] * s) for c in range(3)])\n        # E[d' Oi d] = sum_ce Oi_ce (w_c w_e S0 - W_c w_e Cq_c - w_c W_e Cq_e + W_c W_e Q_ce)\n        Mq = Oi * Q\n        bq = (Oi * Cq[:, None]) @ w\n        W = np.linalg.solve(Mq, bq)\n        val = float(w @ (Oi * S0) @ w - 2 * W @ bq + W @ Mq @ W)\n        return 0.5 * val, W\n\n    grid = np.linspace(-R_MAX, R_MAX, 67)\n    vals = [at(R)[0] for R in grid]\n    k = int(np.argmin(vals))\n    lo, hi = grid[max(k - 1, 0)], grid[min(k + 1, len(grid) - 1)]\n    from scipy.optimize import minimize_scalar\n    res = minimize_scalar(lambda R: at(R)[0], bounds=(lo, hi), method=\"bounded\", options={\"xatol\": 1e-6})\n    R = float(res.x) if res.fun <= vals[k] else float(grid[k])\n    rate, W = at(R)\n    return {\"rate\": float(rate), \"R\": R, \"W\": W.tolist(), \"rows\": int(n)}\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# 3. Q2 and Q3: ranges over the compositions the record cannot tell apart\n# ----------------------------------------------------------------------------------------------------------------\n\ndef _unpack(v):\n    return R_MAX * np.tanh(v[:3]), v[3:]\n\n\ndef _pack(r, w):\n    return np.r_[np.arctanh(np.clip(np.asarray(r) / R_MAX, -0.999999, 0.999999)), w]\n\n\ndef feasible_set(th, n_rec, hbar):\n    A1 = A_of(th)\n\n    def cons(v):\n        r, w = _unpack(v)\n        H = th[\"kappa\"] * w / (1 - r)\n        return np.r_[J0 - n_rec * kl_rate(th, r, w, A1), hbar - H, hbar + H]\n    return cons\n\n\ndef ranges(th, n_rec, hbar, seed=SEED, n_random=N_RANDOM_STARTS):\n    \"\"\"Min and max, over rules theta' with n_rec * rate(theta, theta') <= J0 and |H'_c| <= hbar, of the foreign-halt\n    effect H'_Foreign and of the uniform-cut effect U' = sum_c H'_c. Starts: the estimate, its five relabelings, and\n    `n_random` random rules pulled into the set. Returns the extremes found and the relabeling values (which lie in the\n    set for every record length, because the rate is exactly 0 there).\"\"\"\n    from scipy.optimize import minimize\n    cons = feasible_set(th, n_rec, hbar)\n    A1 = A_of(th)\n    rng = np.random.default_rng(seed)\n    v0 = _pack(th[\"r\"], th[\"w\"])\n    starts = []\n    for perm in itertools.permutations(range(3)):\n        starts.append(_pack(th[\"r\"][list(perm)], th[\"w\"][list(perm)]))\n    wscale = np.maximum(np.abs(th[\"w\"]), 1e-6)\n    for _ in range(n_random):\n        r = np.clip(th[\"r\"] + rng.normal(0, 0.3, 3), -0.95, 0.95)\n        w = th[\"w\"] + rng.normal(0, 1, 3) * wscale.max()\n        v = _pack(r, w)\n        base = starts[rng.integers(len(starts[:6]))]\n        for _ in range(60):                      # pull toward a relabeling until feasible\n            if np.all(cons(v) >= 0):\n                break\n            v = base + 0.5 * (v - base)\n        starts.append(v)\n    kap = th[\"kappa\"]\n\n    def HF(v):\n        r, w = _unpack(v)\n        return kap * w[0] / (1 - r[0])\n\n    def UU(v):\n        r, w = _unpack(v)\n        return float(np.sum(kap * w / (1 - r)))\n\n    out = {}\n    for name, f in ((\"H_Foreign\", HF), (\"U\", UU)):\n        best = {\"min\": (np.inf, None), \"max\": (-np.inf, None)}\n        for v in starts:\n            for sgn, key in ((1, \"min\"), (-1, \"max\")):\n                try:\n                    res = minimize(lambda q: sgn * f(q), v, method=\"SLSQP\",\n                                   constraints=[{\"type\": \"ineq\", \"fun\": cons}], options={\"maxiter\": 500, \"ftol\": 1e-14})\n                    cand = res.x\n                except Exception:\n                    continue\n                c = cons(cand)\n                if np.all(c >= -1e-12 * max(1.0, J0)):\n                    r, w = _unpack(cand)\n                    if stationary(th, r, w)[0]:\n                        val = f(cand)\n                        if (key == \"min\" and val < best[\"min\"][0]) or (key == \"max\" and val > best[\"max\"][0]):\n                            best[key] = (val, cand)\n        rec = {}\n        for key in (\"min\", \"max\"):\n            val, v = best[key]\n            r, w = _unpack(v) if v is not None else (None, None)\n            rec[key] = float(val)\n            rec[key + \"_rules\"] = {\"r\": None if r is None else r.tolist(), \"w\": None if w is None else w.tolist(),\n                                   \"rate\": None if r is None else kl_rate(th, r, w, A1)}\n        out[name] = rec\n    H = effects(th)[\"H\"]\n    out[\"relabeling_values_H_Foreign\"] = sorted(float(h) for h in H)\n    out[\"n_rec\"] = float(n_rec); out[\"hbar\"] = float(hbar); out[\"J0\"] = J0; out[\"n_starts\"] = len(starts)\n    return out\n\n\ndef hbar_of(th):\n    m = 1 - float(np.sum(th[\"a\"]))\n    return (m, False) if m >= 0.01 else (0.01, True)\n\n\ndef measure(prep, mask, dealer=\"DealerTotal\", delay=DELAY, p=P_LAGS, B_=B, n_random=N_RANDOM_STARTS, days_per_year=None,\n            label=\"primary\"):\n    eqs = build_equations(prep, mask, delay, p, dealer)\n    th, stats, ci, nb = estimate(eqs, p, delay, B_)\n    st, minroot = stationary(th)\n    eff = effects(th)\n    dpy = days_per_year if days_per_year else float(mask.sum()) / 4.0\n    n_pooled = float(eqs[\"price\"][\"n\"].sum())\n    per_stock = eqs[\"price\"][\"ok\"].sum(1)\n    n_stock = float(np.median(per_stock[per_stock > 0]))\n    N_st = eqs[\"price\"][\"n_stocks\"]\n    # Q1\n    q1p = q1_price(th)\n    S, ok = tag_filters(prep, mask, th, dealer)\n    # residual covariance of the three rule equations on common rows\n    res = []\n    common = ok.copy()\n    for c in CLASSES:\n        common &= eqs[c][\"ok\"]\n    for c in CLASSES:\n        b = np.array(stats[c][\"coef\"])\n        e = eqs[c][\"yd\"] - eqs[c][\"Xd\"] @ b\n        res.append(e[common])\n    Om = np.cov(np.vstack(res))\n    q1t = q1_tags(th, S, ok, Om)\n    q1 = {\"price_rate_per_stock_day\": q1p[\"rate\"], \"closest_single_memory_price\": q1p,\n          \"tag_rate_per_stock_day\": q1t[\"rate\"], \"closest_single_memory_tags\": q1t,\n          \"information_ratio_tags_to_prices\": (q1t[\"rate\"] / q1p[\"rate\"]) if q1p[\"rate\"] > 0 else float(\"inf\"),\n          \"pinsker_stock_days_prices\": pinsker_days(q1p[\"rate\"]), \"pinsker_stock_days_tags\": pinsker_days(q1t[\"rate\"]),\n          \"years_prices_one_stock_at_least\": pinsker_days(q1p[\"rate\"]) / dpy,\n          \"years_prices_whole_panel_at_least\": pinsker_days(q1p[\"rate\"]) / (dpy * N_st),\n          \"years_tags_one_stock_at_least\": pinsker_days(q1t[\"rate\"]) / dpy,\n          \"residual_cov\": Om.tolist()}\n    hbar, flagged = hbar_of(th)\n    q23_pooled = ranges(th, n_pooled, hbar, n_random=n_random)\n    q23_stock = ranges(th, n_stock, hbar, n_random=n_random)\n    hbar_tight = 2 * float(np.max(np.abs(eff[\"H\"])))\n    q23_tight = ranges(th, n_pooled, hbar_tight, n_random=n_random)\n    return {\"label\": label, \"dealer_series\": dealer, \"delay\": delay, \"p_lags\": p, \"theta\": {\n                \"a\": th[\"a\"].tolist(), \"kappa\": th[\"kappa\"], \"r\": th[\"r\"].tolist(), \"w\": th[\"w\"].tolist()},\n            \"stationary\": st, \"min_root_modulus\": minroot, \"effects\": {\"H\": eff[\"H\"].tolist(), \"U\": eff[\"U\"], \"S\": eff[\"S\"]},\n            \"equations\": stats, \"bootstrap\": {\"B_used\": nb, \"mean_block\": MEAN_BLOCK, \"seed\": SEED, \"ci\": ci},\n            \"records\": {\"pooled_stock_days\": n_pooled, \"median_stock_days\": n_stock, \"n_stocks\": N_st,\n                        \"days_per_year\": dpy},\n            \"hbar\": hbar, \"hbar_floor_used\": flagged, \"Q1\": q1, \"Q2Q3_whole_panel\": q23_pooled,\n            \"Q2Q3_one_stock\": q23_stock, \"hbar_tight\": hbar_tight, \"Q2Q3_whole_panel_tight_box\": q23_tight}\n"
assert hashlib.sha256(EXECUTOR_SRC.encode()).hexdigest() == EXECUTOR_SHA256, "executor changed"
assert hashlib.sha256(CORE_SRC.encode()).hexdigest() == CORE_SHA256, "P1M core changed"
open("p1t_executor.py", "w").write(EXECUTOR_SRC); open("p1m_core.py", "w").write(CORE_SRC)
sys.path.insert(0, "."); import p1t_executor as ex, p1m_core as pm

try:
    from google.colab import drive; drive.mount("/content/drive"); ROOT = "/content/drive/MyDrive/P1T_raw"
except Exception: ROOT = "P1T_raw"
for sub in ("T86", "MI", "TWT49U", "FMTQIK"): os.makedirs(os.path.join(ROOT, sub), exist_ok=True)
START, DEV_END = "20130102", "20161230"
H = {"User-Agent": "Mozilla/5.0 (academic research; P1T)"}
MAN = os.path.join(ROOT, "manifest.jsonl")
def now(): return datetime.datetime.now(datetime.timezone.utc).isoformat()
def fetch(path, url, params, want_date=None):
    fn = os.path.join(ROOT, path + ".json.gz")
    if os.path.exists(fn):
        return json.loads(gzip.open(fn).read())
    print("not on Drive, downloading", path, flush=True)
    for attempt in range(6):
        try:
            r = requests.get(url, params=params, headers=H, timeout=60)
            if r.status_code == 200:
                j = r.json()
                if want_date is not None and j.get("stat") == "OK" and j.get("date") != want_date:
                    raise SystemExit(f"DATE MISMATCH {path}: asked {want_date}, got {j.get('date')}")
                gzip.open(fn, "wb").write(r.content)
                with open(MAN, "a") as m:
                    m.write(json.dumps({"path": path, "url": r.url, "sha256": hashlib.sha256(r.content).hexdigest(), "utc": now()}) + "\n")
                time.sleep(3.5)
                return j
        except SystemExit: raise
        except Exception as e: print("retry", path, repr(e))
        time.sleep(30 * (attempt + 1))
    raise RuntimeError("failed " + path)

# 1. development-period trading calendar (2013-2016 only)
days = []
for y in range(2013, 2017):
    for mth in range(1, 13):
        j = fetch(f"FMTQIK/{y}{mth:02d}", "https://www.twse.com.tw/rwd/zh/afterTrading/FMTQIK",
                  {"date": f"{y}{mth:02d}01", "response": "json"})
        for row in j.get("data", []):
            dg = re.sub(r"[^0-9]", "", row[0]); d = str(int(dg[:-4]) + 1911) + dg[-4:]
            if START <= d <= DEV_END: days.append(d)
days = sorted(set(days)); assert days[-1] <= DEV_END
print(len(days), "development trading days", days[0], days[-1])
# 2. daily T86 and MI_INDEX (already on Drive from the P1T runs)
for k, d in enumerate(days):
    fetch(f"T86/{d}", "https://www.twse.com.tw/rwd/zh/fund/T86", {"date": d, "selectType": "ALLBUT0999", "response": "json"}, d)
    fetch(f"MI/{d}", "https://www.twse.com.tw/rwd/zh/afterTrading/MI_INDEX", {"date": d, "type": "ALLBUT0999", "response": "json"}, d)
# 3. ex-rights table
ev = set()
for y in range(2013, 2017):
    for mth in range(1, 13):
        a = f"{y}{mth:02d}01"; b = f"{y}{mth:02d}{calendar.monthrange(y, mth)[1]:02d}"
        j = fetch(f"TWT49U/{y}{mth:02d}", "https://www.twse.com.tw/rwd/zh/exRight/TWT49U", {"startDate": a, "endDate": b, "response": "json"})
        ev |= ex.parse_twt49u(j)
def load(sub, d): return json.loads(gzip.open(os.path.join(ROOT, sub, d + ".json.gz")).read())

# ---- P1M-A1 part B: decoupled rules (protocol P1M_A1_protocol.md) ----
A1_SHA256 = "11a99f597491d30e294fef3cd92e8ec01f86ff0090faea14dca4fb0364aeb264"
A1_SRC = "\"\"\"P1M-A1: the two analyses the simulated referees asked for (protocol P1M_A1_protocol.md).\n\nTaiwan development period 2013-01-02 to 2016-12-30 only, as in P1M. Builds on p1m_core.py, which is imported\nunchanged.\n\nPart A (no new data; input is the P1M estimate stored in results/p1m_results.json)\n    Q2 and Q3 of P1M recomputed on classes of compositions that the paper's theory covers. The memories r'_c lie in\n    [0, 0.99]. The weights w'_c carry signs:\n      variant \"P\": every weight is nonnegative, which is the positive class of the lower bounds;\n      variant \"T\": each weight has the sign of its tagged estimate (Foreign and Trust nonnegative, Dealers nonpositive).\n    The memories are at least gamma apart. The indistinguishable set, the boxes and the stationarity requirement are\n    those of P1M.\nPart B (the TWSE panel; run in Colab)\n    Decoupled rules: u_{c,t} = sum_{j=1..J} beta_{c,j} x_{t-j} + e_{c,t}, with AR(p) non-feedback flow\n    e_{c,t} = sum_l phi_{c,l} e_{c,t-l} + eta_{c,t}. The feedback kernel no longer decays at the persistence of the\n    class's own flow. Stock fixed effects; iterated Cochrane-Orcutt in moment form; the price equation of P1M; a joint\n    stationary date bootstrap with the same dates for every equation.\n    H_c = kappa * sum_j beta_{c,j}.\n\"\"\"\nfrom __future__ import annotations\n\nimport itertools\n\nimport numpy as np\n\nimport p1m_core as pm\n\nCLASSES = pm.CLASSES\n# ---- Part A --------------------------------------------------------------------------------------------------------\nVARIANTS = {\"P\": (1, 1, 1), \"T\": (1, 1, -1)}     # +1: weight >= 0, -1: weight <= 0 (Foreign, Trust, Dealers)\nGAMMAS = (0.0, 0.01, 0.02, 0.05, 0.10)\nR_MIN = 0.0\nSCAN_GAMMAS = (0.0, 0.05)\nSCAN_FACTORS = (1.0, 5.0, 10.0)                  # times max_c |H_c|; 2 (tight box) and the primary box are in the grid\nTOL_CONS = 1e-12\nTOL_BOUND = 1e-10\n# ---- Part B --------------------------------------------------------------------------------------------------------\nB_RUNS = ((\"primary\", 10, 2), (\"J5\", 5, 2), (\"J20\", 20, 2), (\"AR1\", 10, 1))    # (label, J lags of x, AR order)\nCO_TOL = 1e-12\nCO_MAXIT = 500\nMIN_ROWS = 50\n\n\n# =====================================================================================================================\n# Part A: ranges over a constrained class of compositions\n# =====================================================================================================================\n\ndef _signs(variant):\n    return np.array(VARIANTS[variant], float)\n\n\ndef _project(r, w, variant, gamma):\n    \"\"\"Clip to the bounds and spread the sorted memories so that neighbours are at least gamma apart.\"\"\"\n    s = _signs(variant)\n    w = np.where(s > 0, np.maximum(w, 0.0), np.minimum(w, 0.0))\n    r = np.clip(np.asarray(r, float), R_MIN, pm.R_MAX)\n    order = np.argsort(r, kind=\"stable\")\n    rs = r[order].copy()\n    for k in range(1, 3):\n        rs[k] = max(rs[k], rs[k - 1] + gamma)\n    if rs[2] > pm.R_MAX:\n        rs -= rs[2] - pm.R_MAX\n    rs = np.maximum(rs, R_MIN)\n    out = np.empty(3)\n    out[order] = rs\n    return out, w, order\n\n\ndef in_class(r, w, variant, gamma, hbar, kappa, tol=TOL_BOUND):\n    s = _signs(variant)\n    r, w = np.asarray(r, float), np.asarray(w, float)\n    if np.any(r < R_MIN - tol) or np.any(r > pm.R_MAX + tol) or np.any(s * w < -tol):\n        return False\n    rs = np.sort(r)\n    if np.any(np.diff(rs) < gamma - tol):\n        return False\n    H = kappa * w / (1 - r)\n    return bool(np.all(np.abs(H) <= hbar + tol))\n\n\ndef _constraint_fns(th, n_rec, hbar, gamma, order, A1, with_kl):\n    kap = th[\"kappa\"]\n\n    def sep(v):\n        r = v[:3]\n        return np.array([r[order[1]] - r[order[0]] - gamma, r[order[2]] - r[order[1]] - gamma])\n\n    def box(v):\n        r, w = v[:3], v[3:]\n        H = kap * w / (1 - r)\n        return np.r_[hbar - H, hbar + H] / max(hbar, 1e-12)\n\n    def kl(v):\n        return np.array([(pm.J0 - n_rec * pm.kl_rate(th, v[:3], v[3:], A1)) / pm.J0])\n\n    cons = [{\"type\": \"ineq\", \"fun\": sep}, {\"type\": \"ineq\", \"fun\": box}]\n    if with_kl:\n        cons.append({\"type\": \"ineq\", \"fun\": kl})\n    return cons, sep, box, kl\n\n\ndef _bounds(variant):\n    s = _signs(variant)\n    return [(R_MIN, pm.R_MAX)] * 3 + [((0.0, None) if si > 0 else (None, 0.0)) for si in s]\n\n\ndef _feasible(th, v, variant, gamma, hbar, n_rec, A1, order):\n    r, w = v[:3], v[3:]\n    if not in_class(r, w, variant, gamma, hbar, th[\"kappa\"]):\n        return False\n    if n_rec * pm.kl_rate(th, r, w, A1) > pm.J0 * (1 + TOL_CONS):\n        return False\n    return pm.stationary(th, r, w)[0]\n\n\ndef _starts(th, variant, gamma, seed, n_random):\n    \"\"\"The estimate's six relabelings and n_random random rules, projected onto the class (P1M's start design).\"\"\"\n    rng = np.random.default_rng(seed)\n    pts = []\n    for perm in itertools.permutations(range(3)):\n        pts.append((th[\"r\"][list(perm)], th[\"w\"][list(perm)]))\n    wscale = np.maximum(np.abs(th[\"w\"]), 1e-6).max()\n    for _ in range(n_random):\n        r = np.clip(th[\"r\"] + rng.normal(0, 0.3, 3), -0.95, 0.95)\n        w = th[\"w\"] + rng.normal(0, 1, 3) * wscale\n        pts.append((r, w))\n    out = []\n    for r, w in pts:\n        r2, w2, order = _project(r, w, variant, gamma)\n        out.append((np.r_[r2, w2], order))\n    return out\n\n\ndef relabelings_in_class(th, variant, gamma, hbar):\n    \"\"\"Halt effects of the estimate's relabelings that lie in the class. Their divergence rate is exactly 0, so they lie\n    in the indistinguishable set at every record length (the exact barrier within the class).\"\"\"\n    vals = set()\n    H = pm.effects(th)[\"H\"]\n    for perm in itertools.permutations(range(3)):\n        r, w = th[\"r\"][list(perm)], th[\"w\"][list(perm)]\n        if in_class(r, w, variant, gamma, hbar, th[\"kappa\"]):\n            vals.add(float(th[\"kappa\"] * w[0] / (1 - r[0])))\n    return sorted(vals), bool(in_class(th[\"r\"], th[\"w\"], variant, gamma, hbar, th[\"kappa\"])), H.tolist()\n\n\ndef constrained_ranges(th, n_rec, hbar, variant, gamma, seed=pm.SEED, n_random=pm.N_RANDOM_STARTS):\n    \"\"\"Min and max of H'_Foreign and of U' = sum_c H'_c over compositions theta' in the class (variant, gamma) with\n    n_rec * rate(theta, theta') <= J0, |H'_c| <= hbar and a stationary price process.\n\n    Phase 1 from every start: minimise the divergence within the class (bounds, separation in the start's ordering of\n    memories, box). Starts that reach the indistinguishable set are kept. Phase 2 from each kept start: SLSQP for each\n    extreme with all constraints. Returns the extremes found; the true range contains them. If no start reaches the set,\n    the set is reported empty together with the smallest n * rate found (the true set may still be nonempty).\"\"\"\n    from scipy.optimize import minimize\n    A1 = pm.A_of(th)\n    kap = th[\"kappa\"]\n    bnds = _bounds(variant)\n    feasible_starts = []\n    best_phase1 = np.inf\n    for v0, order in _starts(th, variant, gamma, seed, n_random):\n        cons1, sep, box, kl = _constraint_fns(th, n_rec, hbar, gamma, order, A1, with_kl=False)\n        obj = lambda v: n_rec * pm.kl_rate(th, v[:3], v[3:], A1) / pm.J0\n        try:\n            res = minimize(obj, v0, method=\"SLSQP\", bounds=bnds, constraints=cons1,\n                           options={\"maxiter\": 500, \"ftol\": 1e-16})\n            v1 = res.x\n        except Exception:\n            continue\n        val = n_rec * pm.kl_rate(th, v1[:3], v1[3:], A1)\n        if np.all(sep(v1) >= -TOL_BOUND) and np.all(box(v1) >= -TOL_BOUND) and in_class(v1[:3], v1[3:], variant, gamma, hbar, kap):\n            best_phase1 = min(best_phase1, val)\n            if _feasible(th, v1, variant, gamma, hbar, n_rec, A1, order):\n                feasible_starts.append((v1, order))\n\n    def HF(v):\n        return float(kap * v[3] / (1 - v[0]))\n\n    def UU(v):\n        return float(np.sum(kap * v[3:] / (1 - v[:3])))\n\n    out = {\"variant\": variant, \"gamma\": gamma, \"hbar\": float(hbar), \"n_rec\": float(n_rec), \"J0\": pm.J0,\n           \"n_starts\": 6 + n_random, \"n_feasible_starts\": len(feasible_starts),\n           \"smallest_n_rate_phase1\": float(best_phase1)}\n    rel, est_in, _ = relabelings_in_class(th, variant, gamma, hbar)\n    out[\"relabeling_values_H_Foreign_in_class\"] = rel\n    out[\"estimate_in_class\"] = est_in\n    if not feasible_starts:\n        out[\"empty\"] = True\n        return out\n    out[\"empty\"] = False\n    # every feasible point visited (the kept starts and every feasible SLSQP outcome of either search) is pooled, and\n    # both effects are extremised over the pool: each point is a member of the set, so the pool's extremes are found\n    # values of the true range\n    pool = [v for v, _ in feasible_starts]\n    for name, f in ((\"H_Foreign\", HF), (\"U\", UU)):\n        for v0, order in feasible_starts:\n            cons2, sep, box, kl = _constraint_fns(th, n_rec, hbar, gamma, order, A1, with_kl=True)\n            for sgn in (1, -1):\n                try:\n                    res = minimize(lambda q: sgn * f(q) / max(hbar, 1e-12), v0, method=\"SLSQP\", bounds=bnds,\n                                   constraints=cons2, options={\"maxiter\": 500, \"ftol\": 1e-14})\n                    cand = res.x\n                except Exception:\n                    continue\n                if _feasible(th, cand, variant, gamma, hbar, n_rec, A1, order):\n                    pool.append(cand)\n    out[\"n_pool\"] = len(pool)\n    for name, f in ((\"H_Foreign\", HF), (\"U\", UU)):\n        vals = np.array([f(v) for v in pool])\n        rec = {}\n        for key, k in ((\"min\", int(np.argmin(vals))), (\"max\", int(np.argmax(vals)))):\n            v = pool[k]\n            rec[key] = float(vals[k])\n            rec[key + \"_rules\"] = {\"r\": v[:3].tolist(), \"w\": v[3:].tolist(),\n                                   \"n_rate\": float(n_rec * pm.kl_rate(th, v[:3], v[3:], A1))}\n        out[name] = rec\n    return out\n\n\ndef part_a_configs(th, records, hbar_primary, hbar_tight):\n    \"\"\"The pre-specified configurations (protocol section 3).\"\"\"\n    n_panel, n_stock = records[\"pooled_stock_days\"], records[\"median_stock_days\"]\n    hmax = float(np.max(np.abs(pm.effects(th)[\"H\"])))\n    cfg = []\n    for variant in (\"P\", \"T\"):\n        for gamma in GAMMAS:\n            for box_name, hb in ((\"primary\", hbar_primary), (\"tight\", hbar_tight)):\n                cfg.append({\"key\": f\"{variant}_g{gamma:.2f}_{box_name}_panel\", \"variant\": variant, \"gamma\": gamma,\n                            \"box\": box_name, \"hbar\": hb, \"record\": \"whole_panel\", \"n_rec\": n_panel})\n    for gamma in SCAN_GAMMAS:\n        for f in SCAN_FACTORS:\n            cfg.append({\"key\": f\"T_g{gamma:.2f}_x{f:g}_panel\", \"variant\": \"T\", \"gamma\": gamma, \"box\": f\"{f:g}xmaxH\",\n                        \"hbar\": f * hmax, \"record\": \"whole_panel\", \"n_rec\": n_panel})\n    for variant in (\"P\", \"T\"):\n        cfg.append({\"key\": f\"{variant}_g0.00_primary_stock\", \"variant\": variant, \"gamma\": 0.0, \"box\": \"primary\",\n                    \"hbar\": hbar_primary, \"record\": \"one_stock\", \"n_rec\": n_stock})\n    return cfg\n\n\n# =====================================================================================================================\n# Part B: decoupled rules\n# =====================================================================================================================\n\ndef decoupled_moments(prep, mask, flow, J, p_ar, delay=pm.DELAY):\n    \"\"\"Per-date cross moments of z_t = (u_t, u_{t-1..t-p}, x_{t-delay}, ..., x_{t-delay-(J+p-1)}), within-stock\n    demeaned over the rows where every component is finite (stock means over the full sample; fixed effects are not\n    re-estimated within bootstrap draws, as in P1M).\"\"\"\n    x = prep[\"x\"]\n    N, T = x.shape\n    base = prep[\"usable\"] & pm.universe_mask(prep, T) & mask[None, :]\n    comps = [pm.lag(flow, k) for k in range(p_ar + 1)] + [pm.lag(x, delay + j) for j in range(J + p_ar)]\n    Z = np.stack(comps, -1)\n    ok = base & np.isfinite(Z).all(-1)\n    cnt = ok.sum(1)\n    ok &= (cnt >= MIN_ROWS)[:, None]\n    cnt = ok.sum(1)\n    c = np.maximum(cnt, 1)\n    Zm = np.where(ok[..., None], Z, 0.0).sum(1) / c[:, None]\n    Zd = np.where(ok[..., None], Z - Zm[:, None, :], 0.0)\n    M = np.einsum(\"itk,itl->tkl\", Zd, Zd)\n    return {\"M\": M, \"n\": ok.sum(0), \"n_stocks\": int((cnt > 0).sum()), \"J\": J, \"p\": p_ar, \"k\": Z.shape[-1]}\n\n\ndef _cu(k, p, phi):\n    v = np.zeros(k)\n    v[0] = 1.0\n    v[1:p + 1] = -phi\n    return v\n\n\ndef _cx(k, p, phi, j):\n    \"\"\"Quasi-differenced x_{t-j} (j = 1..J): x_{t-j} - sum_l phi_l x_{t-j-l}.\"\"\"\n    v = np.zeros(k)\n    b = p + 1 + (j - 1)\n    v[b] = 1.0\n    for l in range(1, p + 1):\n        v[b + l] -= phi[l - 1]\n    return v\n\n\ndef _ce(k, p, J, beta, l):\n    \"\"\"e_{t-l} = u_{t-l} - sum_j beta_j x_{t-j-l}.\"\"\"\n    v = np.zeros(k)\n    v[l] = 1.0\n    for j in range(1, J + 1):\n        v[p + 1 + (j - 1) + l] -= beta[j - 1]\n    return v\n\n\ndef co_fit(M, J, p, phi0=None, tol=CO_TOL, maxit=CO_MAXIT):\n    \"\"\"Iterated Cochrane-Orcutt for u = beta(L) x + e, phi(L) e = eta, from the summed moment matrix M.\"\"\"\n    k = M.shape[0]\n    phi = np.zeros(p) if phi0 is None else np.array(phi0, float)\n    converged = False\n    it = 0\n    for it in range(1, maxit + 1):\n        Cx = np.stack([_cx(k, p, phi, j) for j in range(1, J + 1)], 1)\n        beta = np.linalg.solve(Cx.T @ M @ Cx, Cx.T @ M @ _cu(k, p, phi))\n        D = np.stack([_ce(k, p, J, beta, l) for l in range(1, p + 1)], 1)\n        d0 = _ce(k, p, J, beta, 0)\n        phi_new = np.linalg.solve(D.T @ M @ D, D.T @ M @ d0)\n        if np.max(np.abs(phi_new - phi)) < tol:\n            phi = phi_new\n            converged = True\n            break\n        phi = phi_new\n    Cx = np.stack([_cx(k, p, phi, j) for j in range(1, J + 1)], 1)\n    beta = np.linalg.solve(Cx.T @ M @ Cx, Cx.T @ M @ _cu(k, p, phi))\n    resid = _cu(k, p, phi) - Cx @ beta\n    ssr = float(resid @ M @ resid)\n    return beta, phi, ssr, it, converged\n\n\ndef decoupled(prep, mask, J, p_ar, dealer=\"DealerTotal\", B_=pm.B, mean_block=pm.MEAN_BLOCK, seed=pm.SEED, label=\"\"):\n    \"\"\"Point estimates and the joint stationary date bootstrap of beta_c, phi_c, kappa, H_c = kappa sum_j beta_{c,j}\n    and U = sum_c H_c. The price equation is P1M's (p = 5 lags, impact on U_{t-1}).\"\"\"\n    flows = pm.class_flows(prep, dealer)\n    eqs = pm.build_equations(prep, mask, pm.DELAY, pm.P_LAGS, dealer)\n    price = eqs[\"price\"]\n    kappa = float(pm.ols_from(price)[0][pm.P_LAGS])\n    mom = {c: decoupled_moments(prep, mask, flows[c], J, p_ar) for c in CLASSES}\n    point = {}\n    for c in CLASSES:\n        M = mom[c][\"M\"].sum(0)\n        beta, phi, ssr, it, conv = co_fit(M, J, p_ar)\n        n = int(mom[c][\"n\"].sum())\n        point[c] = {\"beta\": beta.tolist(), \"sum_beta\": float(beta.sum()), \"phi\": phi.tolist(),\n                    \"sigma2_eta\": ssr / max(n - J - p_ar - mom[c][\"n_stocks\"], 1), \"rows\": n,\n                    \"stocks\": mom[c][\"n_stocks\"], \"iterations\": it, \"converged\": conv}\n    H = np.array([kappa * point[c][\"sum_beta\"] for c in CLASSES])\n    T = len(price[\"n\"])\n    for c in CLASSES:\n        assert mom[c][\"M\"].shape[0] == T\n    rng = np.random.default_rng(seed)\n    draws, n_fail = [], 0\n    for _ in range(B_):\n        idx = pm.stationary_indices(T, mean_block, rng)\n        try:\n            kb = float(pm.ols_from(price, idx)[0][pm.P_LAGS])\n            sb = []\n            for c in CLASSES:\n                beta, phi, _, _, conv = co_fit(mom[c][\"M\"][idx].sum(0), J, p_ar, phi0=point[c][\"phi\"])\n                sb.append(beta.sum())\n                sb.append(phi[0])\n            sb = np.array(sb)\n        except np.linalg.LinAlgError:\n            n_fail += 1\n            continue\n        Hb = kb * sb[0::2]\n        draws.append(np.r_[sb[0::2], sb[1::2], kb, Hb, Hb.sum()])\n    draws = np.array(draws)\n    names = [f\"sum_beta_{c}\" for c in CLASSES] + [f\"phi1_{c}\" for c in CLASSES] + [\"kappa\"] + \\\n            [f\"H_{c}\" for c in CLASSES] + [\"U\"]\n    ptv = np.r_[[point[c][\"sum_beta\"] for c in CLASSES], [point[c][\"phi\"][0] for c in CLASSES], kappa, H, H.sum()]\n    ci = {nm: {\"est\": float(v), \"ci95\": [float(np.percentile(draws[:, j], 2.5)), float(np.percentile(draws[:, j], 97.5))],\n               \"se\": float(np.std(draws[:, j]))} for j, (nm, v) in enumerate(zip(names, ptv))}\n    return {\"label\": label, \"J\": J, \"ar_order\": p_ar, \"dealer_series\": dealer, \"kappa\": kappa, \"rules\": point,\n            \"H\": H.tolist(), \"U\": float(H.sum()), \"bootstrap\": {\"B_used\": len(draws), \"failed\": n_fail,\n                                                                \"mean_block\": mean_block, \"seed\": seed, \"ci\": ci}}\n"
assert hashlib.sha256(A1_SRC.encode()).hexdigest() == A1_SHA256, "P1M-A1 core changed"
open("p1m_a1_core.py", "w").write(A1_SRC)
import importlib; import p1m_a1_core as a1; importlib.reload(a1)
RES = os.path.join(ROOT, "p1m_a1_partB_results.json")
if os.path.exists(RES):
    resultsB = json.load(open(RES))
    print("P1M-A1 part B was already computed on", resultsB["computed_utc"], "- loaded from Drive, NOT recomputed.")
else:
    assert days and max(days) <= DEV_END, "a date after the development period would be loaded"
    day_files = [(d, load("T86", d), load("MI", d)) for d in days]
    P = ex.build_panel(day_files, ev)
    dates = np.array(P.dates); dev = np.ones(len(dates), bool)
    prep = ex.prepare(P, dev)
    print("panel:", len(P.codes), "codes,", len(dates), "days; universe", len(prep["U"]), flush=True)
    t0 = time.time()
    runs = {}
    for label, J, p in a1.B_RUNS:
        runs[label] = a1.decoupled(prep, dev, J, p, label=label)
        print(label, "done", round(time.time() - t0), "s", flush=True)
    p1m_path = os.path.join(ROOT, "p1m_results.json")
    restricted = None
    if os.path.exists(p1m_path):
        q = json.load(open(p1m_path))["runs"]["primary"]
        restricted = {"kappa": q["theta"]["kappa"], "H": q["effects"]["H"], "U": q["effects"]["U"],
                      "ci": {k: q["bootstrap"]["ci"][k] for k in ("H_Foreign", "H_Trust", "H_Dealers", "U")}}
    resultsB = {"spec": "P1M-A1 part B", "protocol": "P1M_A1_protocol.md", "period": [days[0], days[-1]],
                "n_days": len(days), "executor_sha256": EXECUTOR_SHA256, "core_sha256": CORE_SHA256,
                "a1_core_sha256": A1_SHA256, "computed_utc": now(), "runs": runs, "restricted_P1M": restricted,
                "kappa_matches_P1M": None if restricted is None else abs(runs["primary"]["kappa"] - restricted["kappa"]) < 1e-12}
    json.dump(resultsB, open("p1m_a1_partB_results.json", "w"), indent=1, ensure_ascii=False, default=float)
    with zipfile.ZipFile("p1m_a1_partB_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write("p1m_a1_partB_results.json")
    import shutil
    shutil.copy("p1m_a1_partB_results.json", RES)
    shutil.copy("p1m_a1_partB_results.zip", os.path.join(ROOT, "p1m_a1_partB_results.zip"))

def _f(v):
    return f"{v:.3g}" if isinstance(v, (int, float)) and v == v else str(v)
print("kappa matches P1M:", resultsB["kappa_matches_P1M"])
if resultsB["restricted_P1M"]:
    rc = resultsB["restricted_P1M"]["ci"]
    print("restricted (P1M): H_Foreign", _f(rc["H_Foreign"]["est"]), [_f(v) for v in rc["H_Foreign"]["ci95"]])
for name, R in resultsB["runs"].items():
    ci = R["bootstrap"]["ci"]
    print(f"\n=== {name} === J={R['J']} AR({R['ar_order']}) kappa={_f(R['kappa'])} draws={R['bootstrap']['B_used']}")
    for c in ("Foreign", "Trust", "Dealers"):
        ru = R["rules"][c]
        print(f"  {c}: sum beta {_f(ru['sum_beta'])}  phi {[_f(v) for v in ru['phi']]}  converged {ru['converged']}  rows {ru['rows']}")
        print(f"     H_{c} = {_f(ci['H_'+c]['est'])}  95% [{_f(ci['H_'+c]['ci95'][0])}, {_f(ci['H_'+c]['ci95'][1])}]")
    print(f"  U = {_f(ci['U']['est'])}  95% [{_f(ci['U']['ci95'][0])}, {_f(ci['U']['ci95'][1])}]")
try:
    from google.colab import files; files.download("p1m_a1_partB_results.zip")
except Exception:
    print("saved p1m_a1_partB_results.zip (also on Drive in P1T_raw)")
